# AssureX Claim Engine - Python Classification Model
Trains and compares 3 classification algorithms (Random Forest, Gradient
Boosting, Logistic Regression) on the synthetic warranty claim dataset.
Selects the best-performing model based on validation macro F1-score,
then reports final unbiased performance on the held-out test set.

import matplotlib
matplotlib.use("Agg")  # headless backend — no display needed, just saves files

In [1]:
import json
import os
from datetime import datetime

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, f1_score, precision_score,
                              recall_score)
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler

os.makedirs("model", exist_ok=True)
os.makedirs("reports", exist_ok=True)

## 0. Fix working directory

VS Code's Jupyter extension runs this notebook with `notebooks/` as the
working directory, not the project root — so every relative path below
(`data/processed/...`, `model/...`, `reports/...`) would otherwise fail.
This matches the convention used in `train_model.py`, which is always run
as `python src/ml/train_model.py` from the project root. This cell finds
the root by locating `requirements.txt` and `cd`s there, so every path
after this cell resolves the same way it does in the script.

In [2]:
from pathlib import Path

def find_project_root(marker="requirements.txt"):
    path = Path.cwd()
    for parent in [path] + list(path.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Could not find project root (missing {marker})")

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
print("Working directory set to:", os.getcwd())

Working directory set to: /home/hashirkhan/Downloads/assurex-claim-engine


## 1. Load train / validation / test splits

In [3]:
train_df = pd.read_csv("data/processed/train.csv")
val_df = pd.read_csv("data/processed/val.csv")
test_df = pd.read_csv("data/processed/test.csv")

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

Train: 1050 | Val: 225 | Test: 225


## 2. Feature selection and encoding

Categorical fields are label-encoded; boolean fields are cast to int.
Free-text `fault_description` and identifier/date columns are excluded —
their signal (warranty status, ambiguity) is already captured in derived
fields (`product_age_days`, `warranty status via dates`, etc.)

In [4]:
CATEGORICAL_FEATURES = ["product_category", "fault_type", "damage_type"]
BOOLEAN_FEATURES = ["has_receipt", "has_warranty_card", "has_product_image",
                     "serial_number_match", "repair_authorized",
                     "previous_replacement", "is_duplicate_claim"]
NUMERIC_FEATURES = ["product_age_days", "warranty_duration_days", "repair_count",
                     "purchase_price"]

TARGET = "class_label"

encoders = {}
for col in CATEGORICAL_FEATURES:
    le = LabelEncoder()
    le.fit(pd.concat([train_df[col], val_df[col], test_df[col]]))
    encoders[col] = le

target_encoder = LabelEncoder()
target_encoder.fit(train_df[TARGET])
encoders["class_label"] = target_encoder


def prepare_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in CATEGORICAL_FEATURES:
        out[col] = encoders[col].transform(out[col])
    for col in BOOLEAN_FEATURES:
        out[col] = out[col].astype(int)
    return out[CATEGORICAL_FEATURES + BOOLEAN_FEATURES + NUMERIC_FEATURES]


X_train = prepare_features(train_df)
y_train = target_encoder.transform(train_df[TARGET])

X_val = prepare_features(val_df)
y_val = target_encoder.transform(val_df[TARGET])

X_test = prepare_features(test_df)
y_test = target_encoder.transform(test_df[TARGET])

print("Feature columns:", list(X_train.columns))
print("Classes:", list(target_encoder.classes_))

Feature columns: ['product_category', 'fault_type', 'damage_type', 'has_receipt', 'has_warranty_card', 'has_product_image', 'serial_number_match', 'repair_authorized', 'previous_replacement', 'is_duplicate_claim', 'product_age_days', 'warranty_duration_days', 'repair_count', 'purchase_price']
Classes: ['Invalid', 'ManualReview', 'Valid']


Scale numeric features (helps Logistic Regression converge; harmless for
tree-based models).

In [5]:
scaler = StandardScaler()
scaler.fit(X_train[NUMERIC_FEATURES])

for df_ in (X_train, X_val, X_test):
    df_[NUMERIC_FEATURES] = scaler.transform(df_[NUMERIC_FEATURES])

## 3. Define candidate models with hyperparameter grids

Small grids are used (time-boxed competition setting) but each model still
undergoes genuine tuning via cross-validated grid search on the training set.

In [6]:
model_grids = {
    "RandomForest": {
        "estimator": RandomForestClassifier(random_state=42),
        "param_grid": {
            "n_estimators": [100, 200],
            "max_depth": [8, 12, None],
        },
    },
    "GradientBoosting": {
        "estimator": GradientBoostingClassifier(random_state=42),
        "param_grid": {
            "n_estimators": [100, 150],
            "learning_rate": [0.05, 0.1],
        },
    },
    "LogisticRegression": {
        "estimator": LogisticRegression(max_iter=1000, random_state=42),
        "param_grid": {
            "C": [0.1, 1.0, 10.0],
        },
    },
}

## 4. Train + cross-validate + tune each model

In [7]:
results = {}
fitted_models = {}

for name, cfg in model_grids.items():
    print(f"\n=== {name} ===")

    grid = GridSearchCV(
        cfg["estimator"], cfg["param_grid"],
        cv=5, scoring="f1_macro", n_jobs=-1
    )
    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_
    fitted_models[name] = best_model

    cv_scores = cross_val_score(best_model, X_train, y_train, cv=5, scoring="f1_macro")

    val_preds = best_model.predict(X_val)
    val_acc = accuracy_score(y_val, val_preds)
    val_f1 = f1_score(y_val, val_preds, average="macro")
    val_precision = precision_score(y_val, val_preds, average="macro")
    val_recall = recall_score(y_val, val_preds, average="macro")

    results[name] = {
        "best_params": grid.best_params_,
        "cv_f1_macro_mean": cv_scores.mean(),
        "cv_f1_macro_std": cv_scores.std(),
        "val_accuracy": val_acc,
        "val_f1_macro": val_f1,
        "val_precision_macro": val_precision,
        "val_recall_macro": val_recall,
    }

    print(f"Best params: {grid.best_params_}")
    print(f"CV F1 (macro): {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
    print(f"Val Accuracy: {val_acc:.4f} | Val F1 (macro): {val_f1:.4f}")


=== RandomForest ===
Best params: {'max_depth': None, 'n_estimators': 100}
CV F1 (macro): 0.8500 (+/- 0.0299)
Val Accuracy: 0.8800 | Val F1 (macro): 0.8828

=== GradientBoosting ===
Best params: {'learning_rate': 0.1, 'n_estimators': 150}
CV F1 (macro): 0.9143 (+/- 0.0189)
Val Accuracy: 0.9156 | Val F1 (macro): 0.9152

=== LogisticRegression ===
Best params: {'C': 1.0}
CV F1 (macro): 0.7474 (+/- 0.0327)
Val Accuracy: 0.7511 | Val F1 (macro): 0.7537


## 5. Model comparison summary

In [8]:
comparison_df = pd.DataFrame(results).T
comparison_df = comparison_df.sort_values("val_f1_macro", ascending=False)
print(comparison_df)

comparison_df.to_csv("reports/model_comparison_report.csv")

                                                    best_params  \
GradientBoosting    {'learning_rate': 0.1, 'n_estimators': 150}   
RandomForest           {'max_depth': None, 'n_estimators': 100}   
LogisticRegression                                   {'C': 1.0}   

                   cv_f1_macro_mean cv_f1_macro_std val_accuracy val_f1_macro  \
GradientBoosting           0.914306        0.018869     0.915556     0.915196   
RandomForest               0.850036         0.02995         0.88     0.882787   
LogisticRegression          0.74739        0.032711     0.751111     0.753687   

                   val_precision_macro val_recall_macro  
GradientBoosting              0.927037         0.915556  
RandomForest                  0.907778             0.88  
LogisticRegression            0.787671         0.751111  


## 6. Select best model (by validation macro F1)

In [9]:
best_model_name = comparison_df.index[0]
best_model = fitted_models[best_model_name]
print(f"\nSelected best model: {best_model_name}")


Selected best model: GradientBoosting


## 7. Final unbiased evaluation on held-out TEST set

In [10]:
test_preds = best_model.predict(X_test)
test_probs = best_model.predict_proba(X_test)

test_acc = accuracy_score(y_test, test_preds)
test_f1_macro = f1_score(y_test, test_preds, average="macro")

print(f"\n=== FINAL TEST RESULTS ({best_model_name}) ===")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Macro F1: {test_f1_macro:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, test_preds, target_names=target_encoder.classes_))


=== FINAL TEST RESULTS (GradientBoosting) ===
Test Accuracy: 0.9244
Test Macro F1: 0.9238

Classification Report:
              precision    recall  f1-score   support

     Invalid       0.99      1.00      0.99        75
ManualReview       0.98      0.80      0.88        75
       Valid       0.83      0.97      0.90        75

    accuracy                           0.92       225
   macro avg       0.93      0.92      0.92       225
weighted avg       0.93      0.92      0.92       225



## 8. Confusion matrix (test set)

In [11]:
cm = confusion_matrix(y_test, test_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=target_encoder.classes_,
            yticklabels=target_encoder.classes_)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"Confusion Matrix - {best_model_name} (Test Set)")
plt.tight_layout()
plt.savefig("reports/confusion_matrix_test.png")
plt.close()
print("Saved confusion matrix to reports/confusion_matrix_test.png")

Saved confusion matrix to reports/confusion_matrix_test.png


## 9. Feature importance (tree-based models only)

In [12]:
if hasattr(best_model, "feature_importances_"):
    importances = pd.Series(best_model.feature_importances_, index=X_train.columns)
    importances = importances.sort_values(ascending=False)

    plt.figure(figsize=(8, 6))
    importances.plot(kind="barh")
    plt.title(f"Feature Importance - {best_model_name}")
    plt.tight_layout()
    plt.savefig("reports/feature_importance.png")
    plt.close()
    print("Saved feature importance to reports/feature_importance.png")
    print(importances)
else:
    print(f"{best_model_name} does not expose feature_importances_ (e.g. Logistic Regression).")

Saved feature importance to reports/feature_importance.png
product_age_days          0.187521
fault_type                0.170716
previous_replacement      0.127455
serial_number_match       0.113983
repair_authorized         0.101723
is_duplicate_claim        0.094354
warranty_duration_days    0.061812
has_warranty_card         0.044311
has_receipt               0.033244
has_product_image         0.031817
purchase_price            0.027214
damage_type               0.002746
product_category          0.002000
repair_count              0.001106
dtype: float64


## 10. Sample test predictions (for report/demo)

In [13]:
sample_output = test_df[["claim_id", "class_label"]].copy()
sample_output["predicted_label"] = target_encoder.inverse_transform(test_preds)
for i, cls in enumerate(target_encoder.classes_):
    sample_output[f"confidence_{cls}"] = test_probs[:, i]

sample_output.to_csv("reports/sample_test_predictions.csv", index=False)
print(sample_output.head(10).to_string(index=False))

 claim_id  class_label predicted_label  confidence_Invalid  confidence_ManualReview  confidence_Valid
CLM-01204 ManualReview    ManualReview            0.002137                 0.978512          0.019351
CLM-00671      Invalid         Invalid            0.555161                 0.269221          0.175618
CLM-00119        Valid           Valid            0.040363                 0.101403          0.858234
CLM-00912      Invalid         Invalid            0.882391                 0.041676          0.075932
CLM-00187        Valid           Valid            0.034412                 0.117957          0.847631
CLM-00593      Invalid         Invalid            0.756496                 0.114482          0.129021
CLM-00574      Invalid         Invalid            0.951940                 0.021237          0.026824
CLM-00085        Valid           Valid            0.047628                 0.116211          0.836161
CLM-00579      Invalid         Invalid            0.969094                 0.00731

## 11. Save model, encoders, scaler, and metadata

In [14]:
joblib.dump(best_model, "model/claim_classifier.pkl")
joblib.dump({"encoders": encoders, "scaler": scaler,
             "feature_columns": list(X_train.columns)}, "model/encoders.pkl")

metadata = {
    "model_name": best_model_name,
    "model_version": "1.0.0",
    "trained_at": datetime.now().isoformat(),
    "best_params": results[best_model_name]["best_params"],
    "test_accuracy": test_acc,
    "test_f1_macro": test_f1_macro,
    "val_accuracy": results[best_model_name]["val_accuracy"],
    "val_f1_macro": results[best_model_name]["val_f1_macro"],
    "cv_f1_macro_mean": results[best_model_name]["cv_f1_macro_mean"],
    "classes": list(target_encoder.classes_),
    "feature_columns": list(X_train.columns),
}

with open("model/model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("\nSaved model/claim_classifier.pkl")
print("Saved model/encoders.pkl")
print("Saved model/model_metadata.json")
print(f"\nFinal Test Accuracy: {test_acc:.4f} (SRS target: >= 0.85)")


Saved model/claim_classifier.pkl
Saved model/encoders.pkl
Saved model/model_metadata.json

Final Test Accuracy: 0.9244 (SRS target: >= 0.85)
